In [1]:
# Cell 1 — Paths + imports
import os, json, math, re
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

from PIL import Image, ImageDraw, ImageFont

OCR_JSON_DIR = Path("outputs/ocr_json_jap")              # your cleaned OCR JSON per page
PAGES_DIR    = Path("data/jap_imgs")                # original page PNGs
OUT_TJSON_DIR = Path("outputs/translated_json_jap")      # translated JSON per page
OUT_RENDER_DIR = Path("outputs/final_pages_bbox_jap")    # rendered pages

OUT_TJSON_DIR.mkdir(parents=True, exist_ok=True)
OUT_RENDER_DIR.mkdir(parents=True, exist_ok=True)

# Pick a font that exists on your machine (Windows examples below)
FONT_CANDIDATES = [
    r"C:\Windows\Fonts\arial.ttf",
    r"C:\Windows\Fonts\calibri.ttf",
    r"C:\Windows\Fonts\times.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
if FONT_PATH is None:
    raise FileNotFoundError("Set FONT_PATH to a valid .ttf on your system.")
print("Using font:", FONT_PATH)

Using font: C:\Windows\Fonts\arial.ttf


In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
def load_json(p: Path) -> Any:
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)
    
def get_items(page_obj: Any) -> List[Dict[str, Any]]:
    # Supports either {"items":[...]} or list directly
    if isinstance(page_obj, list):
        return page_obj
    if isinstance(page_obj, dict):
        for k in ["items", "boxes", "bubbles", "ocr"]:
            if k in page_obj and isinstance(page_obj[k], list):
                return page_obj[k]
    raise ValueError("Unrecognized OCR JSON schema")

In [4]:
def normalize_text(text: Optional[str]) -> str:
    # ✅ None-safe
    if text is None:
        return ""
    text = str(text)

    # basic cleanup (tweak as you like)
    text = text.replace("\u3000", " ")          # full-width space -> space
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
def get_ja_text(it: Dict[str, Any]) -> str:
    for k in ["ja_text", "text", "ocr_text", "jp_text"]:
        if k in it:
            return normalize_text(it[k])
    return ""

def get_conf(it: Dict[str, Any]) -> Optional[float]:
    for k in ["conf", "avg_conf", "confidence", "ocr_conf"]:
        if k in it:
            try:
                return float(it[k])
            except:
                return None
    return None

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
tokenizer.src_lang = "ja"

In [6]:
model = model.to(device)

In [ ]:
def translate_ja_to_en_fb(text: Optional[str], max_in_len: int = 256) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    if tokenizer is None or model is None: # translator_mode == "placeholder" or 
        return f"[EN] {text}"

    import torch
    with torch.no_grad():
        batch = tokenizer(
            text,
            return_tensors="pt"
        ).to(device)

        gen = model.generate(**batch, forced_bos_token_id=tokenizer.get_lang_id("en"))
        out = tokenizer.decode(gen, skip_special_tokens=True)

    return normalize_text(out)

In [10]:
obj = load_json(OCR_JSON_DIR / "pg_1.json")

In [12]:
items = get_items(obj)

In [ ]:
out_items = []
for it in items:
    ja = get_ja_text(it)
    en = translate_ja_to_en_fb(ja) if ja else ""
    new_it = dict(it)
    new_it["ja_text"] = ja
    new_it["en_text"] = en
    c = get_conf(it)
    if c is not None:
        new_it["conf"] = c
    out_items.append(new_it)

In [18]:
out_items

[{'bbox_xyxy': [816, 84, 942, 144],
  'text': '幾少こ',
  'conf': 0.6854667067527771,
  'merged_from': 1,
  'ja_text': '幾少こ',
  'en_text': "['The little.']"},
 {'bbox_xyxy': [820, 670, 949, 767],
  'text': '前1港 年に 程は',
  'conf': 0.9987860654707009,
  'merged_from': 3,
  'ja_text': '前1港 年に 程は',
  'en_text': "['For the first port of the year.']"},
 {'bbox_xyxy': [102, 904, 222, 976],
  'text': 'し停海 そ泊賊',
  'conf': 0.768768799183497,
  'merged_from': 2,
  'ja_text': 'し停海 そ泊賊',
  'en_text': "['Stop the Sea and the Thief.']"},
 {'bbox_xyxy': [452, 1133, 543, 1305],
  'text': '何する気だ お',
  'conf': 0.8683150817425762,
  'merged_from': 2,
  'ja_text': '何する気だ お',
  'en_text': "['What I want to do.']"},
 {'bbox_xyxy': [817, 1176, 876, 1294],
  'text': '風は乗',
  'conf': 0.5469527840614319,
  'merged_from': 1,
  'ja_text': '風は乗',
  'en_text': "['The wind runs.']"},
 {'bbox_xyxy': [74, 1392, 196, 1548],
  'text': '平い村 和たは である つて -',
  'conf': 0.9574441209390668,
  'merged_from': 5,
  'ja_text': '平い村 和たは

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ja-en")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ja-en")

C:\Users\abcdj\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 15145.49it/s]


In [7]:
model = model.to(device)

In [8]:
def translate_ja_to_en_hl(text: Optional[str], max_in_len: int = 256) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    if tokenizer is None or model is None: # translator_mode == "placeholder" or 
        return f"[EN] {text}"

    import torch
    with torch.no_grad():
        batch = tokenizer(
            [text],
            return_tensors="pt"
        ).to(device)

        gen = model.generate(**batch)
        out = tokenizer.decode(gen, skip_special_tokens=True)[0]

    return normalize_text(out)

In [9]:
obj = load_json(OCR_JSON_DIR / "pg_1.json")
items = get_items(obj)
out_items = []
for it in items:
    ja = get_ja_text(it)
    en = translate_ja_to_en_hl(ja) if ja else ""
    new_it = dict(it)
    new_it["ja_text"] = ja
    new_it["en_text"] = en
    c = get_conf(it)
    if c is not None:
        new_it["conf"] = c
    out_items.append(new_it)
out_items

[{'bbox_xyxy': [816, 84, 942, 144],
  'text': '幾少こ',
  'conf': 0.6854667067527771,
  'merged_from': 1,
  'ja_text': '幾少こ',
  'en_text': 'How many?'},
 {'bbox_xyxy': [820, 670, 949, 767],
  'text': '前1港 年に 程は',
  'conf': 0.9987860654707009,
  'merged_from': 3,
  'ja_text': '前1港 年に 程は',
  'en_text': "It's been a year since I left port."},
 {'bbox_xyxy': [102, 904, 222, 976],
  'text': 'し停海 そ泊賊',
  'conf': 0.768768799183497,
  'merged_from': 2,
  'ja_text': 'し停海 そ泊賊',
  'en_text': 'The sea, the sea, the bandits.'},
 {'bbox_xyxy': [452, 1133, 543, 1305],
  'text': '何する気だ お',
  'conf': 0.8683150817425762,
  'merged_from': 2,
  'ja_text': '何する気だ お',
  'en_text': 'What are you doing?'},
 {'bbox_xyxy': [817, 1176, 876, 1294],
  'text': '風は乗',
  'conf': 0.5469527840614319,
  'merged_from': 1,
  'ja_text': '風は乗',
  'en_text': "Wind's on."},
 {'bbox_xyxy': [74, 1392, 196, 1548],
  'text': '平い村 和たは である つて -',
  'conf': 0.9574441209390668,
  'merged_from': 5,
  'ja_text': '平い村 和たは である つて -',
  'en_

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

C:\Users\abcdj\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 15505.63it/s]


In [7]:
model = model.to(device)

In [10]:
def translate_ja_to_en_nllb(text: Optional[str], max_in_len: int = 256) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    if tokenizer is None or model is None: # translator_mode == "placeholder" or 
        return f"[EN] {text}"

    import torch
    with torch.no_grad():
        batch = tokenizer(
            text,
            return_tensors="pt"
        ).to(device)

        gen = model.generate(
            **batch,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=256
        )        
        out = tokenizer.decode(gen, skip_special_tokens=True)[0]

    return normalize_text(out)

In [11]:
obj = load_json(OCR_JSON_DIR / "pg_1.json")
items = get_items(obj)
out_items = []
for it in items:
    ja = get_ja_text(it)
    en = translate_ja_to_en_nllb(ja) if ja else ""
    new_it = dict(it)
    new_it["ja_text"] = ja
    new_it["en_text"] = en
    c = get_conf(it)
    if c is not None:
        new_it["conf"] = c
    out_items.append(new_it)
out_items

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'bbox_xyxy': [816, 84, 942, 144],
  'text': '幾少こ',
  'conf': 0.6854667067527771,
  'merged_from': 1,
  'ja_text': '幾少こ',
  'en_text': 'There are a few small ones.'},
 {'bbox_xyxy': [820, 670, 949, 767],
  'text': '前1港 年に 程は',
  'conf': 0.9987860654707009,
  'merged_from': 3,
  'ja_text': '前1港 年に 程は',
  'en_text': 'In the first year of the port, the number of passengers increased.'},
 {'bbox_xyxy': [102, 904, 222, 976],
  'text': 'し停海 そ泊賊',
  'conf': 0.768768799183497,
  'merged_from': 2,
  'ja_text': 'し停海 そ泊賊',
  'en_text': 'Stop at sea and rob the pirates.'},
 {'bbox_xyxy': [452, 1133, 543, 1305],
  'text': '何する気だ お',
  'conf': 0.8683150817425762,
  'merged_from': 2,
  'ja_text': '何する気だ お',
  'en_text': 'What are you doing?'},
 {'bbox_xyxy': [817, 1176, 876, 1294],
  'text': '風は乗',
  'conf': 0.5469527840614319,
  'merged_from': 1,
  'ja_text': '風は乗',
  'en_text': 'The wind is blowing.'},
 {'bbox_xyxy': [74, 1392, 196, 1548],
  'text': '平い村 和たは である つて -',
  'conf': 0.9574441209390668